#### Read project environment properties data

In [1]:
import os
import uuid

from dotenv import load_dotenv

# Specify the path to your external .env file
dotenv_path = "G:/My Office/Be-Data/projects/LDS/env/lds-dtp-krowten-backend-dev.env"
load_dotenv(dotenv_path=dotenv_path)

print("GOOGLE_PROJECT_ID =", os.getenv("GOOGLE_PROJECT_ID"))
print("FIRESTORE_DATABASE =", os.getenv("FIRESTORE_DATABASE"))
print("GOOGLE_APPLICATION_CREDENTIALS =", os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))

GOOGLE_PROJECT_ID = lds-krowten-dev
FIRESTORE_DATABASE = lds-dtp-krowten-masterdata-dev-v2
GOOGLE_APPLICATION_CREDENTIALS = G:\My Office\Be-Data\projects\LDS\env\google-config\client_secrets.json


In [ ]:
print("hello")

##### Firestore configration -> db

In [2]:
import os
from google.cloud import firestore

google_project_id = os.getenv("GOOGLE_PROJECT_ID")
firestore_database = os.getenv("FIRESTORE_DATABASE")

if google_project_id is None or firestore_database is None:
    raise ValueError("Missing GCP_PROJECT_ID or FIRESTORE_DATABASE in env")

db = firestore.Client(project=google_project_id, database=firestore_database)

##### get collection

In [3]:
video_assets = db.collection('video-assets')

#### Retrieve All Documents in a Collection [video_assets]

In [4]:
video_docs = video_assets.stream()
for doc in video_docs:
    # print(f"{doc.id} => {doc.to_dict()}")
    print(f"{doc.id} => ")

9EV4elDhSYwMOFypGlji => 
BoMotMB9xPb901DUqFw7 => 
HHBugFRfHvYrNxS3yoY2 => 
Ho1GHuM8d6zoAvaMaeiN => 
IUoPZTFtAZHYUmrE7PxM => 
Mnu01yweiiHdAvIfnAMo => 
b4vFvsBfMqIdKzmjmZBl => 
gaSlLkRbDDx0gJP4PRcT => 
kygdLBXwdimDwQRUBxYh => 
x3blLw8E9LAEsxniwjKF => 


#### Retrieve Specific Documents by ID [video_assets]

In [ ]:
doc_ids = ['WxE4PLpqtxxE69cZRstU', 'Wp7vgaeXUdAifky3IT1y', 'VjuVcNPNjO1iFv7MLgRO']
doc_refs = [video_assets.document(doc_id) for doc_id in doc_ids]

# Fetch multiple documents at once
docs = db.get_all(doc_refs)

for doc in docs:
    if doc.exists:
        # print(f"Found: {doc.id} => {doc.to_dict()}")
        print(f"Found: {doc.id} => ")


#### Filtered Document List

**Query operators**

The `where()` method takes three parameters: a field to filter on, a comparison operator, and a value. Core operations support the following comparison operators:

* [ < ] less than
* [ <= ] less than or equal to
* [ == ] equal to
* [ > ] greater than
* [ >= ] greater than or equal to
* [ != ] not equal to
* [ array-contains ]
* [ array-contains-any ]
* [ in ]
* [ not-in ]

In [ ]:
# Query documents where 'tag' field is 25K
from google.api_core import exceptions

try:
    query = video_assets.where('tag', '==', '25K')
    docs = query.stream()
    for doc in docs:
        print(f"Document ID: {doc.id}")
except exceptions.GoogleAPICallError as e:
    print(f"Firestore Error: {e.message}")

In [ ]:
from google.api_core import exceptions
from google.cloud.firestore_v1.base_query import FieldFilter

try:
    query = video_assets.where(filter=FieldFilter("`primary-data-source`.`custom-id`", "==", "BME-TGG-S001-EP004"))
    # query = video_assets.where(filter=FieldFilter("flags.`audio-extraction-complete`", "==", True))
    docs = query.stream()
    for doc in docs:
        print(f"Document ID: {doc.id}")
except exceptions.GoogleAPICallError as e:
    print(f"Firestore Error: {e.message}")

#### Create document with Collection

In Google Cloud Firestore, collections are **not explicitly created**. Instead, a collection is automatically generated the moment you add your first document to

In [ ]:
doc_ref = db.collection("bds_test20260408").document("org")
doc_ref.set({"first": "Ada", "last": "Lovelace", "born": 1815})

#### Read Data:
Retrieve a single document or stream all documents in a collection

In [ ]:
# Get a single document
doc = db.collection("bds_test20260408").document("org").get()
if doc.exists:
    print(f"Document data: {doc.to_dict()}")

# Stream all documents
docs = db.collection("bds_test20260408").stream()
for d in docs:
    print(f"{d.id} => {d.to_dict()}")

#### Update Data:
Modify specific fields without overwriting the entire document.

In [ ]:
db.collection("bds_test20260408").document("org").update({"born": 1816})

#### Batch Update Data:
To perform batch updates in Firestore using Python

##### Implementation Steps
1. **Initialize the Batch:** Create a batch object from your Firestore database instance.
2. **Queue Operations:** Add update() or set() operations to the batch by providing the document reference and the data.
3. **Commit the Batch:** Call commit() to execute all queued operations atomically.


`500 Operation Limit: A single batch can contain a maximum of 500 operations. If you need to update more than 500 documents, you must break them into multiple batches.`

In [ ]:
batch = db.batch()

# Define document references
doc_ref_1 = db.collection("bds_test20260408").document("user_1")
doc_ref_2 = db.collection("bds_test20260408").document("user_2")

# Queue update operations
batch.update(doc_ref_1, {"status": "active", "last_login": firestore.SERVER_TIMESTAMP})
batch.update(doc_ref_2, {"status": "inactive"})

# Commit the changes
batch.commit()
# Asynchronous Support
# await batch.commit()

In [ ]:
items_col = db.document("bds_test20260408").collection("videos")
items = []
def chunker(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

for chunk in chunker(items, 499):
    refs = [items_col.document(video_id) for video_id in chunk]
    snaps = list(db.get_all(refs))

    batch = db.batch()
    ops = 0

    for snap in snaps:
        if not snap.exists:
            continue

        # batch.update(snap.reference, {"scheduled_for": None})
        ops += 1

    if ops > 0:
        batch.commit()

In [8]:
import uuid
items_col = db.collection("bds_test20260408").document('users').collection('videos')
videos = ['3-9Cdpd9s6o', '0N1hOWOmSY4']

def chunker(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


for chunk in chunker(videos, 499):
    refs = [items_col.document(video_id) for video_id in chunk]
    batch = db.batch()
    for it in refs:
        batch.update(it, {'hh': 'dump'})
    batch.commit()

In [12]:
from google.api_core.exceptions import NotFound
tems_col = db.collection("bds_test20260408").document('users').collection('videos')
videos =['3-9Cdpd9s6o', '0N1hOWOmSY4', '0N1hOWOmSYd']

# 1. Initialize BulkWriter
bulk_writer = db.bulk_writer()

# 1. Define an error handler
def handle_write_error(error):
    # error.exception contains the underlying gRPC exception
    if isinstance(error.exception, NotFound) or error.exception.code == 404:
        print(f"Skipping {error.reference.id}: Document does not exist.")
        # Return False to tell BulkWriter NOT to retry this operation
        return False

    print(f"A different error occurred on {error.reference.id}: {error.exception}")
    # Return True to retry transient network errors
    return True

# 2. Attach the handler to your BulkWriter
bulk_writer.on_write_error(handle_write_error)

for video_id in videos:
    doc_ref = items_col.document(video_id)
    # 2. Queue the update. BulkWriter automatically handles batching and sending.
    bulk_writer.update(doc_ref, {'bulk_writer': 'dump'})

# 3. Block until all operations in the queue are completed
bulk_writer.close()

print(f"Successfully updated {len(videos)} documents.")

Successfully updated 3 documents.


#### Delete Data:
Remove a document or a specific field.

##### Delete documents
`db.collection("bds_test20260408").document("org").delete()`

##### Delete fields
```
city_ref = db.collection("bds_test20260408").document("org")
city_ref.update({"first": firestore.DELETE_FIELD})
```

In [ ]:
db.collection("bds_test20260408").document("org").delete()

In [ ]:
city_ref = db.collection("bds_test20260408").document("org")
city_ref.update({"first": firestore.DELETE_FIELD})